# Laboratori 02: Aritmetika kompjuterike, kompleksiteti dhe qëndrueshmëria

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; në pjesën e dytë jepen fragmentet thelbësore në GNU Octave.

Ky laborator nis nga një vëzhgim i thjeshtë: kompjuteri nuk punon me numrat realë idealë të matematikës, por me një bashkësi të fundme numrash të paraqitshëm. Prandaj një program mund të jetë i saktë nga pikëpamja sintaksore dhe, megjithatë, të japë një rezultat numerikisht të dobët. Për të gjykuar një llogaritje duhet t'u përgjigjemi tri pyetjeve:

1. **Sa punë kërkon algoritmi?** — kompleksiteti.
2. **Sa i ndjeshëm është vetë problemi ndaj të dhënave?** — kushtëzimi.
3. **Sa gabim shton mënyra konkrete e llogaritjes?** — qëndrueshmëria.

Ritmi i laboratorit është: **llogarit me dorë → parashiko → ekzekuto → mat → interpreto**. Qelizat nuk duhen ekzekutuar të gjitha menjëherë.


## Organizimi i seancës

| Koha | Veprimtaria | Rezultati i pritshëm |
|---:|---|---|
| 0–20 min | Rrumbullakimi me një model dhjetor të thjeshtë | Kuptimi intuitiv i humbjes së informacionit |
| 20–40 min | `float64`, epsiloni dhe anulimi | Matja e gabimit absolut e relativ |
| 40–65 min | Kompleksiteti me dorë | Formula e numrit të veprimeve dhe rendi asimptotik |
| 65–80 min | Kompleksiteti në kompjuter | Kohëmatja dhe pjerrësia log–log |
| 80–100 min | Kushtëzimi dhe qëndrueshmëria | Dallimi problem–algoritëm |
| 100–112 min | Shembulli i plotë dhe punishte | Krahasimi i dy algoritmeve |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Riprodhimi i rezultateve kryesore |


## Rezultatet e të nxënit

Në fund studenti duhet të jetë në gjendje të:

- shpjegojë pse një numër si $0.1$ zakonisht nuk paraqitet saktësisht në bazën dy;
- llogarisë gabimin absolut dhe relativ;
- dallojë gabimin e rrumbullakimit nga gabimi i këputjes;
- numërojë me dorë veprimet elementare të një algoritmi;
- nxjerrë $T(n)$ dhe ta klasifikojë me $O$, $\Omega$ dhe $\Theta$;
- verifikojë eksperimentalisht kompleksitetin përmes kohëmatjes;
- dallojë kushtëzimin e problemit nga qëndrueshmëria e algoritmit;
- krahasojë dy formula matematikisht ekuivalente, por numerikisht jo të barasvlershme;
- ekzekutojë një skript Python dhe një skript Octave jashtë notebook-ut.


# Pjesa I — Python

## 1. Rrumbullakimi pa kompjuter: një makinë dhjetore imagjinare

Para formatit binar, marrim një makinë shumë të thjeshtë që ruan vetëm **tri shifra domethënëse**. Numri

$$x=12.3456$$

ruhet si $\operatorname{fl}(x)=12.3$. Simboli $\operatorname{fl}$ tregon rezultatin pas rrumbullakimit. Informacioni i hequr nuk mund të rikthehet nga veprimet pasuese.

Gabimi absolut është

$$e_a=|\operatorname{fl}(x)-x|=|12.3-12.3456|=0.0456,$$

ndërsa gabimi relativ është

$$e_r=\frac{e_a}{|x|}\approx 3.69\times10^{-3}.$$

Gabimi absolut ka njësinë e $x$; gabimi relativ është pa përmasa dhe lejon krahasimin e madhësive me shkallë të ndryshme.


### 1.1 Ushtrim me dorë

Në makinën me tri shifra domethënëse llogaritni:

$$a=1.00\times10^4,\qquad b=-1.00\times10^4,\qquad c=1.00.$$

1. Gjeni $\operatorname{fl}(a+b)$.
2. Gjeni $\operatorname{fl}(a+c)$.
3. Krahasoni $(a+b)+c$ me $a+(b+c)$, duke rrumbullakuar pas çdo veprimi.

Qëllimi nuk është të provohet se matematika ka humbur vetinë e shoqërimit. Në aritmetikën reale ajo mbetet e vlefshme. Ajo që ndryshon është zbatimi me saktësi të fundme.


In [ ]:
from decimal import Decimal, getcontext

# Përdorim tri shifra domethënëse për të imituar makinën e vogël dhjetore.
getcontext().prec = 3
a = Decimal('1.00e4')
b = Decimal('-1.00e4')
c = Decimal('1.00')

majtas = (a + b) + c       # rrumbullakimi kryhet pas mbledhjes së parë
djathtas = a + (b + c)     # renditja e veprimeve është e ndryshme

print('(a+b)+c =', majtas)
print('a+(b+c) =', djathtas)
print('A janë të barabarta?', majtas == djathtas)


**Interpretim.** Nëse rezultatet ndryshojnë, ndryshimi vjen nga rrumbullakimi i ndërmjetëm. Në simulime me miliona veprime, renditja e veprimeve mund të ndikojë dukshëm në rezultatin përfundimtar.

## 2. Nga sistemi dhjetor te formati binar

Një numër normal i pikës lundruese shkruhet në trajtën

$$x=(-1)^s\,(1.m)_2\,2^e,$$

ku $s$ përcakton shenjën, $m$ mantisën dhe $e$ eksponentin. Në formatin IEEE 754 `float64`, përdoren 1 bit për shenjën, 11 bite për eksponentin dhe 52 bite të ruajtura për pjesën thyesore të mantisës. Biti i parë i mantisës është implicit, prandaj saktësia efektive është 53 bite.

Thyesa $0.1$ ka zhvillim binar periodik; analogjia dhjetore është $1/3=0.333\ldots$. Prandaj vlera që ruhet është numri binar më i afërt, jo pikërisht $0.1$.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

# repr shfaq një paraqitje të shkurtër; format me 20 shifra zbulon afrimin real.
x = 0.1
print('repr(0.1)          =', repr(x))
print('me 20 shifra      =', format(x, '.20f'))
print('0.1 + 0.2         =', format(0.1 + 0.2, '.20f'))
print('(0.1+0.2) == 0.3 ?', (0.1 + 0.2) == 0.3)
print('math.isclose       =', math.isclose(0.1 + 0.2, 0.3))


### 2.1 Krahasimi i numrave realë në program

Krahasimi `x == y` kërkon barazi bit për bit. Për rezultate numerike përdorim një tolerancë:

$$|x-y|\leq \varepsilon_a+\varepsilon_r|y|,$$

ku $\varepsilon_a$ është toleranca absolute dhe $\varepsilon_r$ toleranca relative. Toleranca nuk duhet zgjedhur verbërisht; ajo duhet lidhur me saktësinë e të dhënave dhe me shkallën e problemit.


In [ ]:
def afersisht_barabarte(x, y, tol_abs=1e-12, tol_rel=1e-9):
    """Krahason dy numra duke kombinuar tolerancën absolute dhe relative."""
    diferenca = abs(x - y)
    kufiri = tol_abs + tol_rel * abs(y)
    return diferenca <= kufiri, diferenca, kufiri

ok, diferenca, kufiri = afersisht_barabarte(0.1 + 0.2, 0.3)
print(f'diferenca = {diferenca:.3e}')
print(f'kufiri    = {kufiri:.3e}')
print('pranohet? =', ok)


## 3. Epsiloni i makinës

Epsiloni i makinës është largësia nga $1$ te numri pasues i paraqitshëm:

$$\varepsilon_{\rm mach}=\min\{\varepsilon>0: \operatorname{fl}(1+\varepsilon)>1\}.$$

Për `float64`, $\varepsilon_{\rm mach}=2^{-52}\approx2.22\times10^{-16}$. Një algoritëm i thjeshtë e gjen duke përgjysmuar $\varepsilon$ derisa $1+\varepsilon/2$ të mos dallohet më nga $1$.


In [ ]:
def epsiloni_me_cikel(dtype=np.float64):
    eps = dtype(1.0)
    hapa = 0
    # Kur gjysma e ardhshme humbet nga rrumbullakimi, kemi arritur kufirin.
    while dtype(1.0) + dtype(eps / 2) > dtype(1.0):
        eps = dtype(eps / 2)
        hapa += 1
    return eps, hapa

for dtype in (np.float32, np.float64):
    eps, hapa = epsiloni_me_cikel(dtype)
    print(dtype.__name__, 'eps =', eps, 'hapa =', hapa,
          'NumPy =', np.finfo(dtype).eps)


### 3.1 Sa kushton algoritmi i epsilonit?

Në çdo përsëritje kryhen një pjesëtim me $2$, një mbledhje dhe një krahasim. Nëse cikli bën $p$ përsëritje, kostoja është afërsisht

$$T(p)=3p+c,$$

ku $c$ përmbledh inicializimin dhe kthimin e rezultatit. Prandaj

$$T(p)\in\Theta(p).$$

Këtu $p$ lidhet me numrin e biteve të mantisës; për një format të caktuar është i vogël dhe i fiksuar. Ky shembull tregon se kompleksiteti përcaktohet ndaj një madhësie hyrëse të zgjedhur qartë.

## 4. Gabimi absolut, relativ dhe përhapja e tij

Nëse $\widetilde{x}=x+\delta x$, atëherë për një funksion $y=f(x)$ linearizimi jep

$$\delta y\approx f'(x)\,\delta x.$$

Duke pjesëtuar me $|f(x)|$:

$$\frac{|\delta y|}{|y|}\approx\left|\frac{x f'(x)}{f(x)}\right|\frac{|\delta x|}{|x|}.$$

Faktori

$$\kappa_f(x)=\left|\frac{x f'(x)}{f(x)}\right|$$

është numri relativ i kushtëzimit. Ai i përket problemit matematikor, jo kodit.


In [ ]:
def gabimet(vlera_e_sakte, vlera_e_llogaritur):
    gabim_abs = abs(vlera_e_llogaritur - vlera_e_sakte)
    gabim_rel = gabim_abs / abs(vlera_e_sakte) if vlera_e_sakte != 0 else np.nan
    return gabim_abs, gabim_rel

e_abs, e_rel = gabimet(np.pi, 3.14)
print(f'Gabimi absolut = {e_abs:.8f}')
print(f'Gabimi relativ = {e_rel:.6%}')


## 5. Anulimi: pse zbritja mund të jetë e rrezikshme

Për $x$ të vogël, $\cos x$ është shumë afër $1$. Në shprehjen

$$1-\cos x$$

zbriten dy numra pothuajse të barabartë. Shifrat e përbashkëta anulohen dhe mbeten pikërisht shifrat më pak të sakta. Identiteti

$$1-\cos x=2\sin^2\left(\frac{x}{2}\right)$$

është matematikisht ekuivalent, por numerikisht më i qëndrueshëm për $x$ të vogël.

Para ekzekutimit: parashikoni se cila formulë do të dështojë e para kur $x\to0$.


In [ ]:
x_test = np.array([1e-2, 1e-4, 1e-6, 1e-8, 1e-10])
formula_drejt = 1.0 - np.cos(x_test)
formula_qendrueshme = 2.0 * np.sin(x_test / 2.0)**2

print('       x       1-cos(x)       2sin²(x/2)')
for x_i, y1, y2 in zip(x_test, formula_drejt, formula_qendrueshme):
    print(f'{x_i:9.1e}  {y1:13.5e}  {y2:13.5e}')


In [ ]:
x = np.logspace(-16, -1, 300)
y_drejt = 1.0 - np.cos(x)
y_qendrueshem = 2.0 * np.sin(x / 2.0)**2

# Vlera e qëndrueshme përdoret si referencë për të matur humbjen relative.
gabimi_rel = np.abs(y_drejt - y_qendrueshem) / y_qendrueshem

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(x, gabimi_rel)
ax.set(xlabel='$x$', ylabel='gabimi relativ',
       title='Humbja e saktësisë te $1-\\cos x$')
ax.grid(True, which='both', alpha=0.3)
plt.show()


## 6. Kompleksiteti i algoritmit: llogaritja me dorë

Kompleksiteti përshkruan si rriten koha dhe kujtesa kur rritet madhësia e hyrjes $n$. Fillojmë me **numërimin e veprimeve**, jo me kronometrin.

### 6.1 Shembulli A: shuma e $n$ numrave

```text
s = 0
për i = 1, ..., n:
    s = s + x[i]
```

Numërojmë:

- 1 inicializim;
- $n$ mbledhje;
- rreth $n$ hapa të ciklit;
- $n+1$ kontrolle të kushtit.

Në një model të thjeshtë,

$$T(n)=1+n+n+(n+1)=3n+2.$$

Kur $n$ bëhet i madh, termi $3n$ dominon dhe konstanta $2$ nuk ndryshon ritmin e rritjes. Prandaj

$$T(n)\in\Theta(n).$$

Shënimi $O(n)$ jep një kufi të sipërm, $\Omega(n)$ një kufi të poshtëm dhe $\Theta(n)$ një kufi të shtrënguar.


In [ ]:
def shuma_me_numerim(x):
    s = 0.0
    numeruesi = 1                 # inicializimi s = 0
    for vlere in x:
        s = s + vlere
        numeruesi += 1            # numërojmë mbledhjen kryesore
    return s, numeruesi

for n in (5, 10, 100):
    rezultat, veprime = shuma_me_numerim(np.ones(n))
    print(f'n={n:3d}: shuma={rezultat:6.1f}, veprime të numëruara={veprime}')


### 6.2 Shembulli B: dy cikle të ndërfutura

Nëse veprimi i brendshëm kryhet për çdo çift $(i,j)$ me $i,j=1,\ldots,n$, ai kryhet

$$\sum_{i=1}^{n}\sum_{j=1}^{n}1=n^2$$

herë. Kështu $T(n)=an^2+bn+c$ dhe $T(n)\in\Theta(n^2)$.

Nëse cikli i brendshëm shkon vetëm deri te $i$, numri i veprimeve është

$$\sum_{i=1}^{n}i=\frac{n(n+1)}{2}=\frac12n^2+\frac12n,$$

që përsëri është $\Theta(n^2)$. Faktori $1/2$ ka rëndësi për kohën reale, por jo për klasën asimptotike.


In [ ]:
def ciftet_trekendeshe(n):
    numeruesi = 0
    for i in range(1, n + 1):
        for j in range(1, i + 1):
            numeruesi += 1        # ky është veprimi që po numërojmë
    return numeruesi

for n in (5, 10, 20):
    numerim = ciftet_trekendeshe(n)
    formule = n * (n + 1) // 2
    print(f'n={n:2d}: cikli={numerim:3d}, formula={formule:3d}')


### 6.3 Shembulli C: cikël me përgjysmim

Në algoritmin

```text
ndërsa n > 1:
    n = n / 2
```

pas $k$ hapash kemi $n/2^k\leq1$. Prandaj

$$2^k\geq n,\qquad k\geq\log_2 n,$$

dhe kompleksiteti është $\Theta(\log n)$. Ky derivim është më informues se memorizimi i një tabele kompleksitetesh.


In [ ]:
def hapa_pergjysmimi(n):
    hapa = 0
    while n > 1:
        n //= 2                    # madhësia e problemit përgjysmohet
        hapa += 1
    return hapa

for n in (8, 16, 1000, 1024):
    print(f'n={n:4d}: hapa={hapa_pergjysmimi(n):2d}, log2(n)={np.log2(n):6.2f}')


## 7. Shembull i plotë: vlerësimi i një polinomi

Le të jetë

$$p(x)=a_0+a_1x+\cdots+a_nx^n.$$

### 7.1 Metoda e drejtpërdrejtë

Nëse çdo fuqi $x^k$ ndërtohet nga fillimi me $k$ shumëzime, numri i shumëzimeve është

$$M_{\rm drejt}(n)=\sum_{k=1}^{n}(k+1)
=\frac{n(n+1)}{2}+n
=\frac{n^2+3n}{2}.$$

Ka edhe $n$ mbledhje. Prandaj $T_{\rm drejt}(n)\in\Theta(n^2)$.

### 7.2 Skema e Horner-it

Polinomi rishkruhet si

$$p(x)=a_0+x\left(a_1+x\left(a_2+\cdots+x(a_{n-1}+xa_n)\right)\right).$$

Horner-i kryen saktësisht $n$ shumëzime dhe $n$ mbledhje:

$$T_{\rm Horner}(n)=2n+c\in\Theta(n).$$

Ky është fillimisht një përfundim matematik. Më pas do ta kontrollojmë në kompjuter.


In [ ]:
def polinom_drejt(a, x):
    rezultat = 0.0
    for k, a_k in enumerate(a):
        fuqi = 1.0
        for _ in range(k):
            fuqi *= x              # x^k ndërtohet nga fillimi
        rezultat += a_k * fuqi
    return rezultat

def polinom_horner(a, x):
    rezultat = 0.0
    for a_k in reversed(a):
        rezultat = rezultat * x + a_k  # një shumëzim dhe një mbledhje
    return rezultat

a = np.array([1.0, -3.0, 2.0, 0.5])
x0 = 1.2
print('drejtpërdrejt =', polinom_drejt(a, x0))
print('Horner         =', polinom_horner(a, x0))
print('NumPy          =', np.polynomial.polynomial.polyval(x0, a))


### 7.3 Verifikimi kompjuterik i kompleksitetit

Koha e murit ndikohet nga sistemi operativ, procesori, cache-i dhe interpretuesi. Prandaj përsërisim matjen dhe marrim minimumin ose medianën. Presim

$$t(n)\approx Cn^p.$$

Duke marrë logaritmin:

$$\log t\approx\log C+p\log n.$$

Pjerrësia $p$ e grafikut log–log duhet të jetë afër $2$ për metodën e drejtpërdrejtë dhe afër $1$ për Horner-in.


In [ ]:
def kohematje(funksioni, *args, perseritje=5):
    kohet = []
    for _ in range(perseritje):
        t0 = perf_counter()
        funksioni(*args)
        kohet.append(perf_counter() - t0)
    return min(kohet)              # zvogëlon ndikimin e ndërprerjeve të rastit

gradet = np.array([20, 40, 80, 160, 320])
kohe_drejt = []
kohe_horner = []
rng = np.random.default_rng(2026)

for n in gradet:
    koef = rng.normal(size=n + 1)
    kohe_drejt.append(kohematje(polinom_drejt, koef, 0.99))
    kohe_horner.append(kohematje(polinom_horner, koef, 0.99))

kohe_drejt = np.array(kohe_drejt)
kohe_horner = np.array(kohe_horner)
print('gradet:', gradet)
print('koha drejt:', kohe_drejt)
print('koha Horner:', kohe_horner)


In [ ]:
# Përshtatja lineare në koordinata log–log vlerëson eksponentin p.
p_drejt = np.polyfit(np.log(gradet), np.log(kohe_drejt), 1)[0]
p_horner = np.polyfit(np.log(gradet), np.log(kohe_horner), 1)[0]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(gradet, kohe_drejt, 'o-', label=f'Drejtpërdrejt, p≈{p_drejt:.2f}')
ax.loglog(gradet, kohe_horner, 's-', label=f'Horner, p≈{p_horner:.2f}')
ax.set(xlabel='grada $n$', ylabel='koha [s]', title='Kompleksiteti i matur')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.show()


**Kujdes interpretues.** Nëse pjerrësitë nuk dalin pikërisht $2$ dhe $1$, kjo nuk e rrëzon analizën matematikore. Për madhësi të vogla dominojnë kostot fikse; për madhësi të mëdha ndikojnë cache-i dhe optimizimet. Kompleksiteti asimptotik përshkruan prirjen kur $n$ rritet.

## 8. Qëndrueshmëria: problem i mirë, algoritëm i keq

Duhet të dallojmë:

- **kushtëzimin**, që pyet sa ndryshon zgjidhja e saktë kur ndryshojnë pak të dhënat;
- **qëndrueshmërinë**, që pyet sa gabim shton algoritmi gjatë llogaritjes.

Një algoritëm është prapa-qëndrueshëm nëse rezultati i llogaritur është zgjidhja e saktë e një problemi me të dhëna pak të perturbura. Në mënyrë skematike,

$$\text{gabimi përpara}\lesssim
\text{kushtëzimi}\times\text{gabimi prapa}.$$

Pra, edhe një algoritëm i mirë nuk mund ta shpëtojë plotësisht një problem shumë të keqkushtëzuar; por një algoritëm i paqëndrueshëm mund të dëmtojë edhe një problem të mirëkushtëzuar.


### 8.1 Shembull i thjeshtë: rrënjët e ekuacionit kuadratik

Për $ax^2+bx+c=0$ formula e zakonshme është

$$x_{1,2}=\frac{-b\pm\sqrt{b^2-4ac}}{2a}.$$

Kur $b>0$ dhe $b^2\gg4ac$, te rrënja me shenjën plus zbriten dy numra pothuajse të barabartë. Përdorim

$$q=-\frac12\left(b+\operatorname{sgn}(b)\sqrt{b^2-4ac}\right),$$

pastaj

$$x_1=\frac{q}{a},\qquad x_2=\frac{c}{q}.$$

Formula e dytë shmang anulimin. Kontrolli më i thjeshtë është identiteti i Vietës $x_1x_2=c/a$.


In [ ]:
def rrenje_zakonshme(a, b, c):
    D = b*b - 4*a*c
    r = np.sqrt(D)
    return (-b + r)/(2*a), (-b - r)/(2*a)

def rrenje_qendrueshme(a, b, c):
    D = b*b - 4*a*c
    r = np.sqrt(D)
    q = -0.5 * (b + np.copysign(r, b))  # zgjedh shenjën pa anulim
    return q/a, c/q

a, b, c = 1.0, 1e8, 1.0
xz = rrenje_zakonshme(a, b, c)
xq = rrenje_qendrueshme(a, b, c)
print('formula e zakonshme:', xz, 'produkt =', xz[0]*xz[1])
print('formula e qëndrueshme:', xq, 'produkt =', xq[0]*xq[1])
print('vlera e pritur c/a =', c/a)


### 8.2 Shembull: shuma dhe renditja e veprimeve

Në shumën $10^{16}+1+1+\cdots+1-10^{16}$, shtesat e vogla mund të humbin kur i shtohen menjëherë numrit të madh. Shuma e kompensuar e Kahan-it mban një korrigjim $c$ për pjesën e humbur. Ajo mbetet $\Theta(n)$ në kohë dhe $\Theta(1)$ në kujtesë, por ka më shumë veprime për element dhe zakonisht gabim shumë më të vogël.


In [ ]:
def shuma_kahan(v):
    s = 0.0
    c = 0.0
    for x in v:
        y = x - c                 # rikthen korrigjimin e humbur
        t = s + y
        c = (t - s) - y           # vlerëson pjesën që rrumbullakimi humbi
        s = t
    return s

v = np.concatenate(([1e16], np.ones(100_000), [-1e16]))
print('sum e Python-it =', sum(v))
print('np.sum          =', np.sum(v))
print('Kahan           =', shuma_kahan(v))
print('math.fsum       =', math.fsum(v))
print('vlera e saktë   =', 100_000.0)


## 9. Punishtja — ndrysho, mat dhe shpjego

Kjo pjesë nuk është një demonstrim pasiv. Për secilin eksperiment:

1. shkruani parashikimin;
2. ndryshoni vetëm një parametër;
3. ruani tabelën ose figurën;
4. shpjegoni mekanizmin, jo vetëm ndryshimin vizual.

Eksperimenti vijues krahason gabimin e shumës së zakonshme dhe të Kahan-it kur rritet $n$. Provoni `float32` dhe `float64`, si dhe renditje të ndryshme të të njëjtëve numra.


In [ ]:
def eksperiment_shumash(fuqite=range(1, 7), dtype=np.float64):
    N = []
    gabim_numpy = []
    gabim_kahan = []
    for k in fuqite:
        n = 10**k
        # Konstruktojmë një rast ku numrat e vegjël konkurrojnë me një numër të madh.
        i_madh = dtype(1e12 if dtype == np.float64 else 1e6)
        v = np.concatenate(([i_madh], np.ones(n, dtype=dtype), [-i_madh]))
        sakte = float(n)
        N.append(n)
        gabim_numpy.append(abs(float(np.sum(v)) - sakte))
        gabim_kahan.append(abs(shuma_kahan(v) - sakte))
    return np.array(N), np.array(gabim_numpy), np.array(gabim_kahan)

N, g_np, g_k = eksperiment_shumash(range(1, 6), np.float64)
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(N, g_np + 1e-30, 'o-', label='np.sum')
ax.loglog(N, g_k + 1e-30, 's-', label='Kahan')
ax.set(xlabel='$n$', ylabel='gabimi absolut', title='Gabimi kundrejt madhësisë')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.show()


### 9.1 Pyetje për diskutim

1. Cili algoritëm ka kompleksitet më të mirë: metoda e drejtpërdrejtë e polinomit apo Horner-i? Tregoni formulat.
2. A do të thotë $\Theta(n)$ se dy algoritme marrin gjithmonë të njëjtën kohë? Pse jo?
3. Si dallohet një problem i keqkushtëzuar nga një algoritëm i paqëndrueshëm?
4. Pse formula $2\sin^2(x/2)$ është më e qëndrueshme se $1-\cos x$?
5. Çfarë fiton dhe çfarë kushton shuma e Kahan-it?

## 10. Ekzekutimi lokal i Python-it

Qeliza vijuese krijon një skript të vogël, të pavarur. Qëllimi është që studenti ta ekzekutojë nga terminali dhe të kuptojë se notebook-u është një mjedis pune, jo vetë programi.


In [ ]:
from pathlib import Path

script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_python = r"""import numpy as np
from time import perf_counter

def horner(a, x):
    y = 0.0
    for ak in reversed(a):
        y = y*x + ak
    return y

for n in [10, 100, 1000, 10000]:
    a = np.ones(n + 1)
    t0 = perf_counter()
    y = horner(a, 0.99)
    dt = perf_counter() - t0
    print(f"n={n:5d}, p(0.99)={y:.8g}, koha={dt:.3e} s")
"""
path_python = script_dir / 'laboratori_02_kompleksiteti_qendrueshmeria.py'
path_python.write_text(script_python, encoding='utf-8')
print('U krijua:', path_python)


Ekzekutoni në terminal:

```bash
python3 scripts_local/laboratori_02_kompleksiteti_qendrueshmeria.py
```

# Pjesa II — GNU Octave

Pjesa Octave përmban fragmentet thelbësore. Ajo nuk përsërit çdo figurë të Python-it, por ruan të njëjtën logjikë: formulë, algoritëm, matje dhe kontroll. Kopjojini blloqet në një skedar `.m` ose në editorin e Octave-s.

## 11. Epsiloni i makinës


```octave
format long g
e = 1.0;
hapa = 0;
while 1.0 + e/2.0 > 1.0
    e = e/2.0;      % përgjysmojmë kandidatin
    hapa = hapa + 1;
end
fprintf('eps = %.18e, hapa = %d\n', e, hapa);
fprintf('eps i Octave-s = %.18e\n', eps);
```


## 12. Anulimi dhe formula e qëndrueshme

```octave
x = logspace(-16, -1, 300);
y1 = 1 - cos(x);                 % formulë me anulim
y2 = 2*sin(x/2).^2;              % formulë numerikisht më e qëndrueshme
gabimi_rel = abs(y1-y2)./abs(y2);

loglog(x, gabimi_rel, 'LineWidth', 1.5);
grid on;
xlabel('x'); ylabel('gabimi relativ');
title('Anulimi te 1-cos(x)');
```

## 13. Numërimi i veprimeve dhe Horner-i

```octave
function y = horner(a, x)
    y = 0.0;
    for k = length(a):-1:1
        y = y*x + a(k);          % 1 shumëzim + 1 mbledhje
    end
end

a = [1, -3, 2, 0.5];
x0 = 1.2;
y = horner(a, x0);
fprintf('p(%.2f) = %.12g\n', x0, y);
```

Me $n+1$ koeficientë, cikli kryhet $n+1$ herë në këtë zbatim që nis me zero; një variant që nis nga koeficienti kryesor kryen saktësisht $n$ shumëzime dhe $n$ mbledhje. Në të dy rastet kompleksiteti është $\Theta(n)$.


## 14. Matja e kompleksitetit në Octave

```octave
gradet = [100, 300, 1000, 3000, 10000];
kohet = zeros(size(gradet));

for j = 1:length(gradet)
    n = gradet(j);
    a = ones(1, n+1);
    tic;
    y = horner(a, 0.99);
    kohet(j) = toc;
end

koef = polyfit(log(gradet), log(kohet), 1);
fprintf('Eksponenti i matur p = %.3f\n', koef(1));
loglog(gradet, kohet, 'o-'); grid on;
xlabel('grada n'); ylabel('koha [s]');
```

## 15. Rrënjët kuadratike të qëndrueshme

```octave
a = 1.0; b = 1e8; c = 1.0;
D = b*b - 4*a*c;
r = sqrt(D);

% Formula e zakonshme
x1z = (-b + r)/(2*a);
x2z = (-b - r)/(2*a);

% Formula e qëndrueshme
q = -0.5*(b + sign(b)*r);
x1q = q/a;
x2q = c/q;

fprintf('produkt zakonshëm  = %.16g\n', x1z*x2z);
fprintf('produkt qëndrueshëm = %.16g\n', x1q*x2q);
fprintf('kontrolli c/a        = %.16g\n', c/a);
```


## 16. Skripti lokal Octave

Ruani fragmentet kryesore në `laboratori_02.m` dhe ekzekutoni:

```bash
octave --quiet laboratori_02.m
```

## 17. Raporti i laboratorit

Raporti i shkurtër duhet të përmbajë:

1. një shembull të rrumbullakimit të llogaritur me dorë;
2. derivimin e $T(n)$ për një cikël linear, një cikël të dyfishtë dhe Horner-in;
3. tabelën e kohëve dhe eksponentët e matur;
4. krahasimin $1-\cos x$ me $2\sin^2(x/2)$;
5. kontrollin e Vietës për rrënjët kuadratike;
6. një paragraf që dallon kushtëzimin nga qëndrueshmëria;
7. komandat dhe daljen e ekzekutimit lokal.

## 18. Kontrolli përfundimtar

- [ ] E shpjegoj me fjalët e mia pse $0.1$ nuk ruhet saktësisht.
- [ ] Llogaris gabimin absolut dhe relativ.
- [ ] Nxjerr formulën $T(n)$ para se të përdor shënimin asimptotik.
- [ ] Dalloj $O$, $\Omega$ dhe $\Theta$.
- [ ] Mat kompleksitetin dhe interpretoj pjerrësinë log–log.
- [ ] Dalloj kushtëzimin e problemit nga qëndrueshmëria e algoritmit.
- [ ] Zgjedh formulën numerikisht më të sigurt dhe e justifikoj.
- [ ] Ekzekutoj kodin Python lokalisht.
- [ ] Riprodhoj fragmentet thelbësore në GNU Octave.
